# gcn_attention — Alaska

论文对应实验：训练/验证/测试独立划分。先设置 GNN_DATA_ROOT；请勿覆盖已有权重。

Paper experiment with disjoint train/validation/test sets. Set GNN_DATA_ROOT first. This notebook performs full training when executed; existing checkpoints are protected.


In [ ]:
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_roots = [p for p in [_start, *_start.parents] if (p / "models").is_dir() and (p / "experiments" / "paths.py").is_file()]
if not _roots:
    raise FileNotFoundError("Start Jupyter from this repository or its subdirectories.")
PACKAGE_ROOT = _roots[0]
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import os
import os
# 在任何 torch.cuda.* 之前设置
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import random
import numpy as np
import torch

def set_seed(seed: int):
    # 1. Python 内建随机模块
    random.seed(seed)                                                      
    # 2. NumPy 随机数
    np.random.seed(seed)                                                   
    # 3. PyTorch CPU 随机数
    torch.manual_seed(seed)                                                
    # 4. PyTorch 所有 GPU 随机数
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)                                   
    # 5. 确保每次返回相同的卷积算法 (禁用 cudnn 自动寻找最快算法)
    torch.backends.cudnn.deterministic = True                              
    torch.backends.cudnn.benchmark = False                                 
    # 6. 导致所有 API 使用确定性算法（PyTorch ≥1.8）
    try:
        torch.use_deterministic_algorithms(True)                           
    except AttributeError:
        pass
    # 7. 可选：去除一些潜在的非确定性
    os.environ["PYTHONHASHSEED"] = str(seed)                                

# 使用示例
RandSeed = 42
set_seed(RandSeed)


In [ ]:
# Import modules

%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from models.gcn_attention import SeismicDataset1, SeismicDataset2, GraphNet 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import shutil

In [ ]:
# Define the data directories

model_name = "gcn_attention"

from experiments.paths import get_data_root, check_input_hashes
cwd = str(get_data_root())
data_dir = os.path.join(cwd, "data_ANCHORAGE_DA")
processed_waveform_dir = os.path.join(data_dir, "waveforms_proc_broadband")

check_input_hashes("Alaska", data_dir)


# Data preparation

In [ ]:
# Fix random seed for reproducibility
# np.random.seed(0)

# Data ranges (min/max)
minlatitude = 59
maxlatitude = 63
minlongitude = -153
maxlongitude = -147
maxdepth = 250e3
minmag = 3
maxmag = 6

# Fraction of the data to be used in training
split = 0.8
val_split = 0.1

As part of the data preparation process, we scale the data within the range of $\pm 1$, and split the data set into a training set and a validation set.

In [ ]:
# Get the event catalogue
catalogue = pd.read_csv(os.path.join(data_dir, "catalogue.csv"))[["lat", "lon", "depth", "mag"]]

# Check which events have data
for i, event in catalogue.iterrows():
    # Event data file
    cat_file = os.path.join(processed_waveform_dir, "%d.npy" % i)
    # If file not exists: remove event from catalogue0
    if not os.path.isfile(cat_file):
        catalogue.drop(index=i, inplace=True)

# Print number of events
print(len(catalogue))        

# Event identifiers
ids = catalogue.index.values
# Uniform weights
weights = np.ones((len(ids), 1))
# Event lat/lon/depth/magnitude
catalogue = catalogue.values

# Scale data
catalogue[:, 0] = (catalogue[:, 0] - minlatitude) / (maxlatitude - minlatitude)
catalogue[:, 1] = (catalogue[:, 1] - minlongitude) / (maxlongitude - minlongitude)
catalogue[:, 2] = catalogue[:, 2] / maxdepth
catalogue[:, 3] = (catalogue[:, 3] - minmag) / (maxmag - minmag)

catalogue = (catalogue - 0.5) * 2

# Concatenate identifiers and weights to event data
catalogue = np.concatenate([ids.reshape(-1, 1), weights.reshape(-1, 1), catalogue], axis=1)

# Randomly split events into train, validation, and test sets
inds = np.arange(catalogue.shape[0])
np.random.shuffle(inds)
N_split = int(split * catalogue.shape[0])
N_val_split = N_split + int(val_split * catalogue.shape[0])

train_inds = inds[:N_split]
val_inds = inds[N_split:N_val_split]
test_inds = inds[N_val_split:]

# Split catalogue
train_catalogue = catalogue[train_inds]
val_catalogue = catalogue[val_inds]
test_catalogue = catalogue[test_inds]

from experiments.paths import check_fixed_split
check_fixed_split("Alaska", train_catalogue[:, 0], val_catalogue[:, 0], test_catalogue[:, 0])

# Check data distributions to ensure that train and validation sets are similar!
# fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(9, 7))
# axes = axes.ravel()

# for i in range(4):
#     ax = axes[i]
#     ax.hist(catalogue[:, i+2], bins=20, density=False)
#     ax.hist(train_catalogue[:, i+2], bins=20, alpha=0.5, density=False)
#     ax.hist(test_catalogue[:, i+2], bins=20, alpha=0.5, density=False)

# plt.tight_layout()    
# plt.show()

In [ ]:
# 读取台站信息
stations = pd.read_csv(os.path.join(data_dir, "stations.csv"))[["code", "lat", "lon"]]

# 归一化经纬度，映射到 [-1, 1]
stations["lat"] = ((stations["lat"] - minlatitude) / (maxlatitude - minlatitude) - 0.5) * 2
stations["lon"] = ((stations["lon"] - minlongitude) / (maxlongitude - minlongitude) - 0.5) * 2

In [ ]:
# Open the event-station lookup file
lookup_file = os.path.join(data_dir, "catalogue_station_lookup_final.pickle")

with open(lookup_file, "rb") as f:
    lookup = pickle.load(f)

In [ ]:
from torch.utils.data import DataLoader

# 参数保持一致
N_sub = 50
N_t = 2048

train_dataset = SeismicDataset2(
    data_dir=processed_waveform_dir,
    catalogue=train_catalogue,
    stations=stations,
    lookup=lookup,
    N_sub=N_sub,
    N_t=N_t,
)

val_dataset = SeismicDataset1(
    data_dir=processed_waveform_dir,
    catalogue=val_catalogue,
    stations=stations,
    lookup=lookup,
    N_sub=N_sub,
    N_t=N_t,
)

test_dataset = SeismicDataset1(
    data_dir=processed_waveform_dir,
    catalogue=test_catalogue,
    stations=stations,
    lookup=lookup,
    N_sub=N_sub,
    N_t=N_t,
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
    worker_init_fn=lambda worker_id: np.random.seed(RandSeed + worker_id))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model construction

In [ ]:
import os

# —— 第一步：拿到“执行上下文”的名字 —— 
try:
    # 如果是 .py 文件直接执行，这里会生效
    base_name = os.path.splitext(os.path.basename(__file__))[0]
except NameError:
    # 如果 NameError，说明没有 __file__ （一般在 Notebook 中）
    try:
        # pip install ipynbname
        import ipynbname
        base_name = ipynbname.name()  
    except Exception:
        # 回退一个默认名，避免出错
        base_name = "notebook"

# —— 第二步：在 save/ 下建立以这个名字命名的子目录 —— 


base_name = "gcn_attention_Alaska"

model_name = "gcn_attention"

from experiments.paths import get_save_root
savedir = str(get_save_root() / "Alaska" / "gcn_attention")
if any(os.path.exists(os.path.join(savedir, n)) for n in ("best-model.pth", "last-model.pth")):
    raise FileExistsError("Existing checkpoint directory; choose another GNN_SAVE_ROOT. No weights were overwritten.")
os.makedirs(savedir, exist_ok=True)

savefile_best = os.path.join(savedir, "best-model.pth")
savefile_last = os.path.join(savedir, "last-model.pth")

# —— 之后的训练逻辑保持不变 —— 
best_val_loss = float("inf")
# … 

In [ ]:

from torchviz import make_dot

# ========== 参数配置 ==========
N_sub = 50
N_t = 1024
params = {
    "N_t": N_t,
    "dropout_rate": 0.15,
    "activation": "relu",
}

# ========== 设置设备 ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========== 实例化模型 ==========
model = GraphNet(activation="relu").to(device)

# ========== 打印结构信息 ==========
print("Model architecture:")
print(model)

# # ========== 用 torchviz 生成计算图 ==========
# dummy_input = (
#     torch.randn(1, N_sub, N_t, 3).to(device),       # waveforms
#     torch.randn(1, N_sub, 1, 3).to(device),         # coords
#     torch.ones(1, N_sub).to(device)                 # weights
# )
# output = model(*dummy_input)

# # 保存结构图为 PNG
# make_dot(output, params=dict(model.named_parameters())).render("graph_model", format="png")
# print("✅ Saved model structure graph to graph_model.png")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# ========== 设备 ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========== 模型 ==========
model = GraphNet(activation="relu").to(device)

# ========== 优化器 / 损失函数 ==========
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # 同 model.LR
# optimizer = optim.Adam(model.parameters(), lr=2e-5)  # 同 model.LR
criterion = nn.L1Loss()  # MAE = Mean Absolute Error

# ========== 日志 writer（可选） ==========
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter(logdir)

# ========== 加载数据生成器 ==========
# train_loader, val_loader 是你写的 PyTorch Dataset / DataLoader 替代原来的 DataGenerator
# 注意：这里假设 train_loader/val_loader 已准备好

# ========== 保存模型 ==========
best_val_loss = float("inf")
# savefile_best = "save/best-model.pth"
# savefile_last = "save/last-model.pth"

# ========== 训练循环 ==========
num_epochs = 50
no_improve_epoch = 70
epoch = 0
no_improve_count = 0

train_loss_list = []
val_loss_list = []

loss_weights = torch.tensor([1.0, 1.0, 0.3, 0.1], device=device)  # 第4个权重较低

train_flag = 1
lr_change = 0
# for epoch in range(num_epochs):
while train_flag == 1:
    if epoch > num_epochs and no_improve_count > no_improve_epoch:
        if lr_change == 0:
            optimizer = optim.Adam(model.parameters(), lr=2e-5)
            no_improve_count = 0
            lr_change = 1
        else:
            train_flag = 0
    model.train()
    train_loss = 0
    for (waveforms, coords, weights), labels, _ in tqdm(train_loader):
        waveforms, coords, weights, labels = (
            waveforms.to(device),
            coords.to(device),
            weights.to(device),
            labels.to(device),
        )

        optimizer.zero_grad()
        preds = model(waveforms, coords, weights)
        # loss = criterion(preds, labels)
        loss_per_dim = torch.abs(preds - labels)    # (B,4)
        weighted_loss = loss_per_dim * loss_weights      # (B,4)
        loss = weighted_loss.mean()   # 也可以用sum，取决于你想用均值还是总和
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ======== 验证 ========
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for (waveforms, coords, weights), labels, _ in val_loader:
            waveforms, coords, weights, labels = (
                waveforms.to(device),
                coords.to(device),
                weights.to(device),
                labels.to(device),
            )
            preds = model(waveforms, coords, weights)
            # loss = criterion(preds, labels)
            loss_per_dim = torch.abs(preds - labels)    # (B,4)
            weighted_loss = loss_per_dim * loss_weights      # (B,4)
            loss = weighted_loss.mean()   # 也可以用sum，取决于你想用均值还是总和
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"[Epoch {epoch+1:03d}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
      # 记录loss
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)

    # writer.add_scalars("Loss", {"Train": train_loss, "Val": val_loss}, epoch)
    epoch += 1
    no_improve_count += 1

    # ========== Checkpoint ==========
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), savefile_best)
        no_improve_count = 0
        print("✅ Saved best model")

    torch.save(model.state_dict(), savefile_last)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# —— 1) 载入最优模型 —— 
# model.load_state_dict(torch.load('./h=3/best-model.pth'))
# model.load_state_dict(torch.load('./h=3/last-model.pth'))
state = torch.load(savefile_best, map_location=device)
print(savefile_best)
model.load_state_dict(state)
model.eval()

# —— 2) 对 train、validation 和 test 一次性做预测 —— 
def collect_preds(loader):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for (waveforms, coords, weights), labels, _ in loader:
            waveforms, coords, weights = (
                waveforms.to(device),
                coords.to(device),
                weights.to(device),
            )
            preds = model(waveforms, coords, weights)
            # print(preds[:,2])
            # print(labels[:,2])
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.vstack(all_preds), np.vstack(all_labels)

train_P, train_L = collect_preds(train_loader)
val_P, val_L = collect_preds(val_loader)
test_P,  test_L  = collect_preds(test_loader)

# —— 3) 还原到原始物理尺度 —— 
def unscale(X):
    # X[:,0]: latitude, X[:,1]: longitude, X[:,2]: depth, X[:,3]: mag
    lat = (X[:,0] / 2 + 0.5) * (maxlatitude - minlatitude) + minlatitude
    lon = (X[:,1] / 2 + 0.5) * (maxlongitude - minlongitude) + minlongitude
    dep = (X[:,2] / 2 + 0.5) * maxdepth / 1000
    mag = (X[:,3] / 2 + 0.5) * (maxmag - minmag) + minmag
    return np.stack([lat, lon, dep, mag], axis=1)

train_P = unscale(train_P)
train_L = unscale(train_L)
val_P   = unscale(val_P)
val_L   = unscale(val_L)
test_P  = unscale(test_P)
test_L  = unscale(test_L)

# —— 4) 计算误差并换算经纬度单位 —— 
def compute_stats(P, L, name):
    abs_err = np.abs(P - L)
    sq_err  = (P - L)**2

    # 经纬度度→公里
    abs_err[:,0] *= 111  # latitude
    abs_err[:,1] *= 54  # longitude
    sq_err[:,0]  *= 111**2
    sq_err[:,1]  *= 54**2

    # MAE 和 MSE 的均值 & 标准差
    mae_m = abs_err.mean(axis=0)
    mae_s = abs_err.std(axis=0)
    mse_m = sq_err.mean(axis=0)
    mse_s = sq_err.std(axis=0)

    labels = ["Latitude", "Longitude", "Depth", "Magnitude"]
    units  = ["km",       "km",        "km",    ""       ]
    print(f"\n—— {name} 数据集 ——")
    for i,(m,s,lab,u) in enumerate(zip(mae_m, mae_s, labels, units)):
        print(f"{lab:9s} MAE = {m:6.2f} ± {s:6.2f} {u}")
    for i,(m,s,lab,u) in enumerate(zip(mse_m, mse_s, labels, units)):
        print(f"{lab:9s} MSE = {m:6.2f} ± {s:6.2f} {u}^2")
    # 计算 R2
    r2s = []
    for i in range(P.shape[1]):
        r2 = r2_score(L[:, i], P[:, i])
        r2s.append(r2)
    
    for r2, lab in zip(r2s, labels):
        print(f"{lab:9s} R2  = {r2:6.3f}")


# —— 5) 打印结果 —— 
compute_stats(train_P, train_L, "训练")
compute_stats(val_P,   val_L,   "验证")
compute_stats(test_P,  test_L,  "测试")
all_P = np.vstack([train_P, val_P, test_P])
all_L = np.vstack([train_L, val_L, test_L])
compute_stats(all_P, all_L, "整体")
